In [1]:
import os
import numpy as np
from osgeo import gdal

def convert_isce_data(input_file, output_root, file_type):
    """
    Reads ISCE output and converts to GeoTIFF (for ML) and PNG (for viewing).
    """
    if not os.path.exists(input_file):
        print(f"Error: File not found: {input_file}")
        return

    print(f"Processing {file_type} from {input_file}...")
    
    # Open the dataset
    ds = gdal.Open(input_file, gdal.GA_ReadOnly)
    if not ds:
        print("Failed to open file.")
        return

    # Read Band 1
    band = ds.GetRasterBand(1)
    
    # --- Data Extraction Logic ---
    # Case A: Complex Data (Real + Imaginary) -> Calculate Magnitude
    if band.DataType in [gdal.GDT_CFloat32, gdal.GDT_CFloat64]:
        print("  - Detected Complex data. Computing Magnitude...")
        complex_data = band.ReadAsArray()
        data = np.abs(complex_data) # Sqrt(Real^2 + Imag^2)
    
    # Case B: Standard Data (e.g. Coherence 0-1)
    else:
        print("  - Detected Real data. Reading directly...")
        data = band.ReadAsArray()

    # Replace NaNs (Not a Number) with 0 for ML safety
    data = np.nan_to_num(data, nan=0.0)

    # --- Output 1: Float32 GeoTIFF (Best for Machine Learning) ---
    # GeoTIFF supports 'Create', so this part remains the same
    driver_tif = gdal.GetDriverByName('GTiff')
    out_tif = f"{output_root}.tif"
    out_ds = driver_tif.Create(out_tif, ds.RasterXSize, ds.RasterYSize, 1, gdal.GDT_Float32)
    
    # Copy georeferencing info
    out_ds.SetGeoTransform(ds.GetGeoTransform())
    out_ds.SetProjection(ds.GetProjection())
    out_ds.GetRasterBand(1).WriteArray(data.astype(np.float32))
    out_ds.GetRasterBand(1).SetNoDataValue(0)
    out_ds.FlushCache()
    out_ds = None
    print(f"  > Saved ML Input: {out_tif}")

    # --- Output 2: Scaled PNG (Best for Human Viewing) ---
    # We stretch the contrast so 0-1 becomes 0-255
    valid_pixels = data[data > 0]
    if valid_pixels.size > 0:
        p2, p98 = np.percentile(valid_pixels, (2, 98))
        
        # Clip data to this range and scale to 0-255
        data_norm = np.clip(data, p2, p98)
        data_norm = (data_norm - p2) / (p98 - p2) * 255.0
        data_norm = data_norm.astype(np.uint8)

        # FIX: Create in Memory first, then CreateCopy to PNG
        mem_driver = gdal.GetDriverByName('MEM')
        mem_ds = mem_driver.Create('', ds.RasterXSize, ds.RasterYSize, 1, gdal.GDT_Byte)
        mem_ds.GetRasterBand(1).WriteArray(data_norm)

        # Write to PNG using CreateCopy
        driver_png = gdal.GetDriverByName('PNG')
        out_png = f"{output_root}.png"
        driver_png.CreateCopy(out_png, mem_ds, strict=0)
        
        mem_ds = None
        print(f"  > Saved Visualization: {out_png}")
    else:
        print("  ! Warning: Data is empty or all zeros. Skipping PNG.")

    print("-" * 30)

if __name__ == "__main__":
    # Define the polarizations to export
    polarizations = ["vv", "vh"]
    
    for pol in polarizations:
        # 1. Define the path based on the sequential processing folders
        # This assumes you renamed your output folders to merged_vv and merged_vh
        input_file = f"merged_{pol}/topophase.cor.geo"
        output_root = f"coherence_{pol}"
        
        print(f"\n--- Exporting {pol.upper()} Polarization ---")
        
        if os.path.exists(input_file):
            # 2. Run the conversion for this polarization
            convert_isce_data(input_file, output_root, f"Coherence ({pol.upper()})")
        else:
            # Fallback: check if files are in a single 'merged' folder with pol suffixes
            alt_path = f"merged/topophase_{pol}.cor.geo"
            if os.path.exists(alt_path):
                convert_isce_data(alt_path, output_root, f"Coherence ({pol.upper()})")
            else:
                print(f"⚠️ Skipping {pol}: File {input_file} not found.")

    print("\n✅ Coherence export complete for all available polarizations.")


--- Exporting VV Polarization ---
Processing Coherence (VV) from merged_vv/topophase.cor.geo...
  - Detected Real data. Reading directly...
  > Saved ML Input: coherence_vv.tif
  > Saved Visualization: coherence_vv.png
------------------------------

--- Exporting VH Polarization ---
Processing Coherence (VH) from merged_vh/topophase.cor.geo...
  - Detected Real data. Reading directly...
  > Saved ML Input: coherence_vh.tif
  > Saved Visualization: coherence_vh.png
------------------------------

✅ Coherence export complete for all available polarizations.
